# 07 · IPC Update from Caller Feedback

Parse Radio Ergo weekly feedback PDFs into impact signals, then adjust IPC food-security phases week by week.

Needs `pip install -e '.[analysis]'`. Place the feedback PDFs in `data/external/radio_ergo_weekly_feedback/`.

In [ ]:
import pandas as pd

from somali_foodsec_radio.config import get_config
from somali_foodsec_radio.feedback import (
    adjust_ipc_phases_with_threshold,
    aggregate_weekly_impact,
    create_impact_signals,
    extract_detailed_calls_from_pdf,
    infer_impact_level,
    plot_time_series,
)
from somali_foodsec_radio.geo import (
    fuzzy_match_locations,
    load_geojson,
    match_location_to_geo_df,
    normalize_location_name,
    plot_ipc_map_single,
)

cfg = get_config()

## 1. Extract calls from every feedback PDF

In [ ]:
pdf_dir = cfg["paths"]["data_external"] / "radio_ergo_weekly_feedback"
all_calls = pd.concat(
    [extract_detailed_calls_from_pdf(p) for p in sorted(pdf_dir.glob("*.pdf"))],
    ignore_index=True,
)
print(f"Extracted {len(all_calls)} calls")
all_calls.head()

## 2. Normalise locations and detect impact signals

In [ ]:
all_calls["location_normalized"] = all_calls["location"].apply(
    normalize_location_name
)
feedback = create_impact_signals(all_calls)
feedback["impact_level"] = feedback.apply(infer_impact_level, axis=1)
feedback.head()

## 3. Match feedback locations to IPC areas

In [ ]:
geo_df = load_geojson(
    str(cfg["paths"]["data_external"] / "Somalia-Somalia IPC Post GU 2024.json")
)
feedback = match_location_to_geo_df(feedback, geo_df)
feedback = fuzzy_match_locations(
    feedback, geo_df, score_cutoff=cfg["geo"]["fuzzy_score_cutoff"]
)
feedback_matched = feedback[feedback["matched_area"].notnull()]

## 4. Aggregate weekly impact

In [ ]:
weekly_impact_df = aggregate_weekly_impact(feedback_matched)
weekly_impact_df.head()

## 5. Plot weekly feedback signals for selected regions

In [ ]:
for area in ["Mudug", "Hiraan", "Sool"]:
    plot_time_series(weekly_impact_df, area=area)

## 6. Adjust IPC phases week by week

In [ ]:
for week in sorted(weekly_impact_df["week_start"].unique()):
    geo_updated = adjust_ipc_phases_with_threshold(
        geo_df, weekly_impact_df, week
    )
    plot_ipc_map_single(geo_updated, week)